# 01 · Setup and data exploration

**Missions covered:** 2 (cache) · 3 (loading a session) · 4 (exploring the laps)

**Goal:** set up the environment, load a Formula 1 session with [FastF1](https://docs.fastf1.dev/)
and understand the structure of the data before any analysis.

**Session studied:** 2024 Monaco Grand Prix qualifying.

**Prerequisites**
- The `.venv` virtual environment is activated and the dependencies from `requirements.txt` are installed.
- Jupyter Lab is started from the root of the repo (the notebook itself runs from `notebooks/`).
- Lap and telemetry data require a (free) Formula 1 account.
  Sign in from a terminal: `python -m fastf1 auth f1tv --authenticate`

**How to use this notebook:** each step starts with a ✏️ **Exercise**. Try to answer it yourself
(add a new cell below it) before reading the ✅ **Solution** and the 💡 **Explanation** that follow.

## Imports

| Library | Role in the project |
|---|---|
| `os` | handle file paths and folders |
| `fastf1` | download official Formula 1 data |
| `fastf1.plotting` | official driver and tyre colours (used from notebook 03) |
| `numpy` | numerical computing (statistics, conversions) |
| `pandas` | work with data tables (*DataFrames*) |
| `matplotlib.pyplot` | draw charts |
| `display` | show a table properly in Jupyter (instead of `print`) |

In [ ]:
import os

import fastf1
import fastf1.plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Mission 2 · Enabling the FastF1 cache

**Why a cache?** Each session weighs several tens of MB. Without a cache, FastF1 would download it again
on every run. With the cache, the first load is slow and the following ones are almost instant.

> ✏️ **Exercise**
>
> Enable the FastF1 cache in a `cache/` folder at the **root of the repo**.
>
> 1. The notebook runs from `notebooks/`. Which **relative** path points to `cache/` at the root?
> 2. Store this path in a **single variable**. Why is that better than writing it twice?
> 3. FastF1 refuses a cache folder that does not exist. Which `os` function creates it, without an error if it already exists?
> 4. Use `os.path.abspath` to display the full path and check it ends with `f1-telemetry-analysis/cache`.
> 5. Enable the cache. Hint: since FastF1 3.9, `enable_cache` is deprecated; look for `fastf1.Cache.configure` in the documentation.
>
> Bonus: add a check that stops the cell if the notebook is **not** running from `notebooks/`.
> Should this check go before or after creating the folder? Why?

In [ ]:
# ✅ Solution
# The relative path "../cache" is only correct if the notebook runs from notebooks/
assert os.path.basename(os.getcwd()) == "notebooks", f"Unexpected working directory: {os.getcwd()}"

cache_dir = os.path.abspath("../cache")
os.makedirs(cache_dir, exist_ok=True)
print("FastF1 cache:", cache_dir)

fastf1.Cache.configure(cache_dir=cache_dir)

💡 **Explanation**

1. From `notebooks/`, `..` is the parent folder (the root of the repo), so the path is `../cache`.
2. The path is used several times. With a single variable, changing it in one place is enough:
   you can never end up with two different paths by mistake.
3. `os.makedirs(path, exist_ok=True)` creates the folder, and `exist_ok=True` avoids an error when it already exists.
4. `os.path.abspath(...)` turns the relative path into a full one, which is easy to check visually.
5. `fastf1.Cache.configure(cache_dir=...)`: the parameter must be passed **by name** (`cache_dir=`),
   because it comes after a `*` in the function definition.

**Bonus:** the `assert` goes **before** `os.makedirs`. If it came after, a `cache/` folder could already have been
created in the wrong place by the time the error is raised.

Thanks to the relative path, the project works wherever the repo is cloned (Linux, macOS, another folder).
The `cache/` folder is listed in `.gitignore`: it never goes to GitHub.

## Helper function · `show`

> ✏️ **Exercise**
>
> `print(df)` displays a table as plain text: columns are cut with `...` and the layout is hard to read.
>
> Write a function `show(df, columns=None, n=None)` that:
> - keeps only the given `columns` (all of them by default);
> - keeps only the first `n` rows (all of them by default);
> - displays the result as a real Jupyter table, without hiding any column.
>
> Hints: `IPython.display.display`, `df.head(n)`, and the pandas option `display.max_columns`
> (look for `pd.option_context`).

In [ ]:
# ✅ Solution
def show(df, columns=None, n=None):
    '''Display a DataFrame as a Jupyter table, without truncated columns.

    columns : list of column names to keep (all by default)
    n       : number of rows to display (all by default)
    '''
    if columns is not None:
        df = df[columns]
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None):
        display(df)

💡 **Explanation**

```python
show(df)                      # the whole table
show(df, n=5)                 # the first 5 rows
show(df, ["col_a", "col_b"])  # only these columns
```

- `columns=None` and `n=None` are **optional** parameters: when they stay `None`, nothing is filtered.
- `pd.option_context("display.max_columns", None)` tells pandas not to hide any column,
  only for the duration of this display (thanks to the `with`).
- The function has no `return`: it returns `None`. That is why `show(df).unique()` fails with
  `'NoneType' object has no attribute 'unique'`.

The function is meant for **DataFrames**. To display a simple result (`.shape`, `.columns`…),
just write it alone on the last line of a cell: Jupyter displays it automatically.

## Mission 3 · Loading a session

> ✏️ **Exercise**
>
> Load the **qualifying** session of the **2024 Monaco Grand Prix**.
>
> 1. Read the documentation of `fastf1.get_session`. Which **three** arguments does it expect?
>    Which values are accepted for the Grand Prix and for the session type?
> 2. `get_session` alone does not download anything. Which method actually loads the data?
>    Why do you think the two steps are separate?
> 3. Store the three parameters in constants at the top of the cell. What is the benefit?
>
> After running the cell, run it again. What do you notice about the loading time? Look inside `cache/`.

In [ ]:
# ✅ Solution
YEAR = 2024
GRAND_PRIX = "Monaco"
SESSION_TYPE = "Q"

session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION_TYPE)
session.load()
print(session.event["EventName"], "-", session.name)

💡 **Explanation**

1. `fastf1.get_session(year, grand_prix, session_type)`:
   - **the year**: `2024`;
   - **the Grand Prix**: a name (`"Monaco"`) or a round number (`8`);
   - **the session type**: `"FP1"`, `"FP2"`, `"FP3"`, `"Q"` (qualifying), `"S"` (sprint), `"R"` (race).
2. `.load()` downloads the data (or reads it from the cache). The steps are separate so you can choose what to load:
   for example `session.load(telemetry=False)` when telemetry is not needed.
3. With constants, analysing another race only means changing these three lines.

The second run is almost instant: the data is read from `cache/`, where FastF1 stores files by
year, Grand Prix and session (`cache/2024/2024-05-26_Monaco_Grand_Prix/...`).

### Checking that the laps are loaded

Without a Formula 1 account sign-in, FastF1 retrieves the results (from another source, Ergast)
but **not** the laps or the telemetry: accessing `session.laps` then raises a `DataNotLoadedError`.
This cell catches the error with `try` / `except` and prints a clear message instead.

In [ ]:
# Without a Formula 1 account sign-in, FastF1 only retrieves the results:
# accessing session.laps then raises a DataNotLoadedError.
try:
    laps_count = len(session.laps)
except fastf1.exceptions.DataNotLoadedError:
    laps_count = 0

if laps_count == 0:
    print("⚠️ No laps loaded. Sign in from a terminal with:")
    print("   python -m fastf1 auth f1tv --authenticate")
else:
    print(f"✅ {laps_count} laps loaded")

### Qualifying classification

> ✏️ **Exercise**
>
> `session.results` has one row per driver and 22 columns. Display a real **qualifying classification**
> with `show`, keeping only the useful columns.
>
> 1. Which columns do you need to know **who** finished **where**, for **which team**, with **which times**?
> 2. Try `show(session.results, "Position")`. Why is the result not a table?
> 3. What does `NaT` mean in the `Q2` and `Q3` columns?
> 4. Look closely at the `Position` column. Is everything consistent?

In [ ]:
# ✅ Solution
RANKING_COLUMNS = ["Position", "Abbreviation", "FullName", "TeamName", "Q1", "Q2", "Q3"]
show(session.results, RANKING_COLUMNS)

💡 **Explanation**

1. Position, driver (`Abbreviation`, `FullName`), team (`TeamName`) and the time of each phase (`Q1`, `Q2`, `Q3`).
2. A **string** selects a single column: `df["col"]` returns a `Series`. A **list** selects a table:
   `df[["col_a", "col_b"]]` returns a `DataFrame`. With a single column, you cannot even tell who is first.
3. `NaT` (*Not a Time*, the equivalent of `NaN` for durations) means the driver did not take part in that phase:
   the last 5 are knocked out in Q1, the next 5 in Q2.
4. ⚠️ **Data issue:** `Position` contains 14 twice and 15 twice, and stops at 18 for 20 drivers.
   The row order is not reliable at the bottom of the table either: MAG has a Q2 time, but appears after
   ALO and SAR, who were knocked out in Q1. The top of the classification (Q3) is consistent, but this column
   should not be trusted blindly for places 14 to 20.

## Mission 4 · Exploring `session.laps`

**Key concept: one row = one lap by one driver.**
Each column describes that lap: its time, the tyre used, its number, pit stops…

Before any analysis, we **look** at the data. Otherwise, we risk computing averages on
missing or abnormal values without noticing.

Documentation: [Timing and Telemetry Data → Laps](https://docs.fastf1.dev/core.html#fastf1.core.Laps).

### 4.1 · Size

> ✏️ **Exercise**
>
> How many rows and columns does `session.laps` have? Is it consistent with the number of drivers?
> Tool: the `.shape` attribute.

In [ ]:
# ✅ Solution
session.laps.shape

💡 **Explanation**

**426 rows** (one per lap) and **31 columns**. With 20 drivers, that is about 21 laps each on average,
but the actual number varies a lot (see 4.6): qualifying has three phases, and a driver knocked out
in Q1 drives much less than a driver who makes it to Q3.

### 4.2 · Columns

> ✏️ **Exercise**
>
> List the columns. Try to guess what each one means **before** checking the documentation.
> Tool: `.columns`.

In [ ]:
# ✅ Solution
session.laps.columns

💡 **Explanation**

The columns fall into a few families:

| Family | Columns | Content |
|---|---|---|
| Identity | `Driver`, `DriverNumber`, `Team` | who drove the lap |
| Times | `LapTime`, `Sector1Time`, `Sector2Time`, `Sector3Time` | duration of the lap and of its three sectors |
| Timing | `Time`, `LapStartTime`, `LapStartDate`, `Sector*SessionTime` | when the lap happened during the session |
| Pit stops | `PitOutTime`, `PitInTime`, `Stint` | pit exits and entries, stint number |
| Tyres | `Compound`, `TyreLife`, `FreshTyre` | tyre compound, age in laps, new tyre or not |
| Speeds | `SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST` | speeds at the measurement points (km/h) |
| Quality | `IsAccurate`, `Deleted`, `DeletedReason`, `FastF1Generated`, `IsPersonalBest` | reliability and validity of the lap |
| Track | `TrackStatus` | flags during the lap (`1` = green, `2` = yellow…) |

### 4.3 · First rows

> ✏️ **Exercise**
>
> Display the first 10 rows with your `show` function (use the `n` parameter **by name**).
>
> Look at Leclerc's lap times: some are around 1:12, others around 1:40 or more. Why?
> Hint: look at the `PitOutTime` and `PitInTime` columns on the same rows.

In [ ]:
# ✅ Solution
show(session.laps, n=10)

💡 **Explanation**

The first 10 rows are all Leclerc's laps. They show an **alternating pattern**:
- lap 1 (1:47): **out-lap** from the pits (`PitOutTime` filled in), the tyres are warmed up;
- laps 2 and 4 (about 1:12): **flying laps**, flat out;
- lap 3 (1:37): **cool-down lap**, driven slowly to prepare the next attempt;
- lap 5 (1:56): **in-lap** to the pits (`PitInTime` filled in).

So in qualifying, most laps are **not** fast laps: this is essential to know before computing an average.

### 4.4 · Data types

> ✏️ **Exercise**
>
> Display the type of each column. Find at least three different types (`object`, `float64`, `timedelta64`…).
>
> 1. Why is `LapTime` not a plain number?
> 2. Why is `LapNumber` a `float` and not an integer?
>
> Tools: `.dtypes` or `.info()`.

In [ ]:
# ✅ Solution
session.laps.dtypes

💡 **Explanation**

| pandas type | Meaning | Examples |
|---|---|---|
| `object` | text | `Driver`, `Compound`, `Team`, `TrackStatus` |
| `float64` | decimal number | `LapNumber`, `TyreLife`, `SpeedST` |
| `bool` | true / false | `IsAccurate`, `Deleted`, `FreshTyre` |
| `timedelta64` | **duration** | `LapTime`, `Sector1Time`, `PitInTime` |
| `datetime64` | real **date and time** | `LapStartDate` |

1. `LapTime` is a duration (minutes, seconds, milliseconds). The `timedelta64` type adds and compares
   durations correctly (`1:12.4 + 0:19.3`). For statistics, it will be converted to seconds with
   `.dt.total_seconds()` (notebook 02).
2. numpy's classic integer type cannot represent a missing value (`NaN`). pandas therefore uses `float` to be safe.

### 4.5 · Missing values

> ✏️ **Exercise**
>
> How many values are missing in each column? Display only the columns with at least one missing value.
>
> Above all, find out **why** they are missing: first lap? pit exit? red flag?
>
> Tool: `.isna().sum()`, then a boolean filter on the result.

In [ ]:
# ✅ Solution
missing = session.laps.isna().sum()
missing[missing > 0]

💡 **Explanation**

Every gap has an explanation:
- **`PitOutTime` and `PitInTime` (334 missing out of 426)**: this is **normal**. These columns only have a value for
  out-laps (92) and in-laps (92). Other laps, by definition, have neither.
- **`LapTime` (79 missing)**: 78 of these are **in-laps**. The driver enters the pit lane before
  the finish line: the lap is never completed, so it has no time. The last case is an isolated anomaly
  (SAI, lap 21).
- **`Sector3Time` (78)** and **`SpeedFL` (92)**: same reason. The third sector and the speed at the finish line
  are not measured when the driver goes into the pits.
- **`Position`**: empty on every lap, because this column only makes sense in a race.

### 4.6 · Key columns for what comes next

> ✏️ **Exercise**
>
> Look at the values taken by the key columns: `Driver`, `Compound`, `IsAccurate`, `Deleted`, `TrackStatus`, `TyreLife`.
>
> 1. How many laps did each driver do? Why do some drivers have almost three times more laps than others?
> 2. Which tyres were used?
> 3. What share of laps is flagged as not accurate?
> 4. How many laps were deleted, and why?
> 5. What does `TrackStatus` = `12` mean?
>
> Tools: `.value_counts()` on **one column** (`df["col"]`), `.describe()`.
> Careful: `.unique()` and `.value_counts()` apply to a column, not to the whole table.

In [ ]:
# ✅ Solution
session.laps["Driver"].value_counts()

In [ ]:
# ✅ Solution
session.laps["Compound"].value_counts()

In [ ]:
# ✅ Solution
session.laps["IsAccurate"].value_counts()

In [ ]:
# ✅ Solution
session.laps["Deleted"].value_counts()

In [ ]:
# ✅ Solution
show(session.laps[session.laps["Deleted"]], ["Driver", "LapNumber", "LapTime", "DeletedReason"], n=5)

In [ ]:
# ✅ Solution
session.laps["TrackStatus"].value_counts()

In [ ]:
# ✅ Solution
session.laps["TyreLife"].describe()

💡 **Explanation**

1. From 30 laps (GAS) down to 11 (ALO, BOT, ZHO). Drivers knocked out in Q1 (ALO, SAR, PER, BOT, ZHO) have 11 or 12 laps;
   those who reach Q3 have between 24 and 30.
2. All 426 laps are on **soft tyres** (`SOFT`): in qualifying, everyone looks for maximum grip.
   `TyreLife` goes from 1 to 12 laps: every attempt is made on almost new tyres.
3. 185 laps out of 426 (43 %) are flagged as **not accurate**. 184 of them are out-laps or in-laps.
4. 21 laps were **deleted** by race control, mostly for exceeding track limits (`TRACK LIMITS AT TURN 10`, `TURN 1`).
5. `1` means green flag. `12` means the status changed from green (`1`) to **yellow flag** (`2`) during the lap:
   47 laps were affected.

## Exploration summary

> ✏️ **Exercise**
>
> Before reading the summary below, write your own in a few bullet points:
> what does `session.laps` contain, which pitfalls did you spot, and which laps should be removed before comparing drivers?

💡 **Summary**

- `session.laps` contains **one row per lap**: 426 laps and 31 columns for the 2024 Monaco qualifying.
- Times are **durations** (`timedelta64`): they must be converted to seconds to compute statistics.
- **Most laps are not fast laps**: out-laps, cool-down laps and in-laps.
- **Missing values are not errors**: almost all of them are in-laps.
- Before any comparison, the data must be **filtered** to remove:
  - laps with a pit stop (`PitInTime`, `PitOutTime`);
  - inaccurate laps (`IsAccurate`);
  - deleted laps (`Deleted`);
  - laps under yellow flag (`TrackStatus`).
- In `session.results`, the `Position` column is **inconsistent** for places 14 to 20.